# Notebook 06 — Mini proyecto EDA: el Titanic 🚢

¡Has llegado al último notebook de la serie! En los anteriores aprendiste, paso a paso, los fundamentos de pandas:

| Notebook | Tema |
|---|---|
| 01 | Series, DataFrames, inspección y acceso |
| 02 | Filtrado, `.loc` y ordenamiento |
| 03 | Valores faltantes (`isna`, `dropna`, `fillna`) |
| 04 | `groupby` y `value_counts` |
| 05 | Combinar DataFrames (`concat`, `merge`) |

Hoy **no aprendes conceptos nuevos**: los **aplicas todos juntos** sobre un dataset clásico del análisis de datos: el **Titanic**.

## ¿De qué va este notebook?

Vas a hacer un **EDA** (*Exploratory Data Analysis* — análisis exploratorio de datos), que es exactamente lo que hace un Data Scientist cuando recibe un dataset nuevo:

1. **Cargar e inspeccionar** los datos.
2. **Limpiar** los valores faltantes.
3. **Filtrar** subconjuntos interesantes.
4. **Agrupar** para descubrir patrones.
5. **Sacar conclusiones**.

## Objetivos de aprendizaje

1. Aplicar todo lo aprendido en NB01–NB05 sobre un dataset nuevo.
2. Tomar decisiones de limpieza con criterio.
3. Responder preguntas de negocio con código pandas.
4. Practicar el flujo de trabajo típico de un EDA.

---

## 1. Setup

Cargamos `titanic` desde seaborn. El dataset contiene información de los pasajeros del trasatlántico que se hundió en 1912.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

df = sns.load_dataset("titanic")
df.head()

### Columnas más importantes

| Columna | Significado |
|---|---|
| `survived` | 1 = sobrevivió, 0 = no sobrevivió |
| `pclass` / `class` | Clase del billete (1ª, 2ª, 3ª) |
| `sex` | Sexo del pasajero |
| `age` | Edad |
| `fare` | Precio del billete |
| `embarked` | Puerto de embarque (S=Southampton, C=Cherbourg, Q=Queenstown) |
| `deck` | Cubierta (muchos NaN) |

---

## 2. Paso 1 — Inspeccionar (NB01)

Antes de tocar los datos, hay que **mirarlos**. ¿Qué tamaño tiene? ¿Qué columnas hay? ¿Qué tipos de dato?

### 🏋️ Ejercicio 1 — Inspección inicial

Asigna las siguientes variables:

- **`titanic_shape`** → tupla `(filas, columnas)` del DataFrame `df`.
- **`titanic_columns`** → lista de **nombres** de todas las columnas de `df`.
- **`numeric_columns`** → lista de **nombres** de las columnas **numéricas** de `df` (usa `select_dtypes`).

💡 Tip: revisa el Notebook 01 si no recuerdas la sintaxis.

In [ ]:
# YOUR CODE HERE
titanic_shape = None
titanic_columns = None
numeric_columns = None


In [ ]:
# Tests
assert titanic_shape == (891, 15), f"Expected (891, 15), got {titanic_shape}"

assert isinstance(titanic_columns, list), "titanic_columns must be a list"
assert len(titanic_columns) == 15, f"Expected 15 columns, got {len(titanic_columns)}"
assert "survived" in titanic_columns, "'survived' must be in titanic_columns"
assert "deck" in titanic_columns, "'deck' must be in titanic_columns"

assert isinstance(numeric_columns, list), "numeric_columns must be a list"
expected_numeric = {"survived", "pclass", "age", "sibsp", "parch", "fare"}
assert set(numeric_columns) == expected_numeric, \
    f"Expected numeric columns {expected_numeric}, got {set(numeric_columns)}"

print("✅ ¡Bien! Inspeccionaste correctamente la estructura del dataset.")
print(f"   Filas: {titanic_shape[0]}, columnas: {titanic_shape[1]}")
print(f"   Numéricas: {numeric_columns}")

---

## 3. Paso 2 — Limpiar valores faltantes (NB03)

Mira primero cuántos NaN hay por columna:

In [ ]:
df.isna().sum()

Decisiones de limpieza (estrategia mixta, como aprendiste en NB03):

| Columna | NaN | Decisión | Razón |
|---|---|---|---|
| `deck` | 688 | **Eliminar la columna** | Demasiados NaN (77%) — imputar inventaría datos |
| `age` | 177 | **Imputar con la mediana** | Solo 20% NaN — la mediana es razonable |
| `embarked` | 2 | **Eliminar esas filas** | Muy pocos casos, no vale la pena imputar |
| `embark_town` | 2 | (se eliminan junto con `embarked`) | Son las mismas 2 filas |

### 🏋️ Ejercicio 2 — Limpieza completa

Crea un `DataFrame` llamado **`df_clean`** que sea el resultado de aplicar las decisiones de la tabla a `df`:

1. Elimina la columna `deck`.
2. Rellena los `NaN` de `age` con la **mediana** de `age` (calculada sobre el `df` original).
3. Elimina las filas que tengan `NaN` en `embarked`.

Al final, `df_clean` no debe tener ningún `NaN` y la columna `deck` no debe existir.

In [ ]:
# YOUR CODE HERE
df_clean = None


In [ ]:
# Tests
assert isinstance(df_clean, pd.DataFrame), "df_clean must be a DataFrame"
assert "deck" not in df_clean.columns, "'deck' column should have been dropped"
assert df_clean.shape == (889, 14), f"Expected shape (889, 14), got {df_clean.shape}"
assert df_clean.isna().sum().sum() == 0, \
    f"df_clean must have zero NaN, found {df_clean.isna().sum().sum()}"

# Age column was imputed (originally 177 NaN, now 0)
assert df_clean["age"].isna().sum() == 0, "'age' column must have no NaN after imputation"
# Median should be preserved
assert np.isclose(df_clean["age"].median(), 28.0, atol=0.5), \
    f"Median age should be ~28.0 after imputation, got {df_clean['age'].median()}"

print("✅ ¡Excelente! Tu DataFrame está limpio.")
print(f"   {df_clean.shape[0]} filas conservadas (de 891 originales).")
print(f"   {df_clean.shape[1]} columnas (eliminamos 'deck').")

---

## 4. Paso 3 — Filtrar (NB02)

Vamos a responder una pregunta concreta:

> *¿Quiénes fueron las **mujeres mayores** de **primera clase** que **sobrevivieron**?*

Necesitas combinar 3 condiciones con `&` y luego ordenar por edad descendente.

### 🏋️ Ejercicio 3 — Filtro multi-condición

Crea un `DataFrame` llamado **`first_class_female_survivors`** con las filas de `df_clean` que cumplan:

- `sex` igual a `"female"`,
- `class` igual a `"First"`,
- `survived` igual a `1`,

ordenadas por `age` de mayor a menor (descendente).

In [ ]:
# YOUR CODE HERE
first_class_female_survivors = None


In [ ]:
# Tests
assert isinstance(first_class_female_survivors, pd.DataFrame), \
    "first_class_female_survivors must be a DataFrame"
assert first_class_female_survivors.shape[0] == 89, \
    f"Expected 89 rows, got {first_class_female_survivors.shape[0]}"
assert (first_class_female_survivors["sex"] == "female").all(), \
    "All rows must have sex == 'female'"
assert (first_class_female_survivors["class"] == "First").all(), \
    "All rows must have class == 'First'"
assert (first_class_female_survivors["survived"] == 1).all(), \
    "All rows must have survived == 1"

# Sort check: age descending
ages = first_class_female_survivors["age"].tolist()
assert ages == sorted(ages, reverse=True), "Rows must be sorted by age descending"
assert first_class_female_survivors["age"].iloc[0] == 63.0, \
    f"Oldest survivor in this group was 63.0 years old, got {first_class_female_survivors['age'].iloc[0]}"

print(f"✅ ¡Bien! Encontraste {first_class_female_survivors.shape[0]} mujeres de 1ª clase que sobrevivieron.")
print(f"   La mayor tenía {first_class_female_survivors['age'].iloc[0]} años.")
first_class_female_survivors.head()

---

## 5. Paso 4 — Agrupar para descubrir patrones (NB04)

Pregunta clave del dataset:

> *¿Cómo varió la **tasa de supervivencia** por **clase** y por **sexo**?*

Truco: como `survived` es 0/1, **el promedio de esa columna es la tasa de supervivencia** del grupo. 🧠

### 🏋️ Ejercicio 4 — Tasa de supervivencia por grupo

Crea una `Series` llamada **`survival_rate_by_class_sex`** que contenga la **tasa de supervivencia promedio** (media de `survived`) **agrupada por `class` y `sex`** (en ese orden).

In [ ]:
# YOUR CODE HERE
survival_rate_by_class_sex = None


In [ ]:
# Tests
assert isinstance(survival_rate_by_class_sex, pd.Series), \
    "survival_rate_by_class_sex must be a pandas Series"
assert isinstance(survival_rate_by_class_sex.index, pd.MultiIndex), \
    "Index must be a MultiIndex (class, sex)"
assert survival_rate_by_class_sex.index.names == ["class", "sex"], \
    f"Index names must be ['class', 'sex'], got {survival_rate_by_class_sex.index.names}"

# Spot-check key values
first_female = survival_rate_by_class_sex.loc[("First", "female")]
assert np.isclose(first_female, 0.967, atol=0.01), \
    f"Female / First survival rate should be ~0.967, got {first_female:.3f}"

third_male = survival_rate_by_class_sex.loc[("Third", "male")]
assert np.isclose(third_male, 0.135, atol=0.01), \
    f"Male / Third survival rate should be ~0.135, got {third_male:.3f}"

second_female = survival_rate_by_class_sex.loc[("Second", "female")]
assert np.isclose(second_female, 0.921, atol=0.01), \
    f"Female / Second survival rate should be ~0.921, got {second_female:.3f}"

print("✅ ¡Genial! Descubriste un patrón clave del dataset.")
print(survival_rate_by_class_sex)
print("\n💡 Las mujeres de 1ª clase sobrevivieron ~7x más que los hombres de 3ª clase.")

---

## 6. Paso 5 — Conteos relativos (NB04)

Última pregunta:

> *¿Qué **proporción** de los pasajeros embarcó en cada puerto?*

Para proporciones (en vez de conteos absolutos) usamos `value_counts(normalize=True)`. La suma de los valores resultantes es **1.0** (= 100%).

### 🏋️ Ejercicio 5 — Proporciones

Crea una `Series` llamada **`embarked_proportions`** con la **proporción** de pasajeros por puerto de embarque (columna `embarked` de `df_clean`).

In [ ]:
# YOUR CODE HERE
embarked_proportions = None


In [ ]:
# Tests
assert isinstance(embarked_proportions, pd.Series), \
    "embarked_proportions must be a pandas Series"
assert len(embarked_proportions) == 3, \
    f"Expected 3 ports, got {len(embarked_proportions)}"
assert set(embarked_proportions.index) == {"S", "C", "Q"}, \
    f"Ports must be S, C, Q — got {set(embarked_proportions.index)}"
assert np.isclose(embarked_proportions.sum(), 1.0, atol=1e-6), \
    f"Proportions must sum to 1.0, got {embarked_proportions.sum()}"

# Most common port should be Southampton (S)
assert embarked_proportions.index[0] == "S", \
    f"Most common port should be 'S' (Southampton), got '{embarked_proportions.index[0]}'"
assert np.isclose(embarked_proportions["S"], 0.7244, atol=0.005), \
    f"S proportion should be ~0.7244, got {embarked_proportions['S']:.4f}"
assert np.isclose(embarked_proportions["Q"], 0.0866, atol=0.005), \
    f"Q proportion should be ~0.0866, got {embarked_proportions['Q']:.4f}"

print("✅ ¡Excelente! Calculaste las proporciones por puerto.")
print(embarked_proportions)
print(f"\n💡 ~72% de los pasajeros embarcó en Southampton.")

---

## 7. Conclusiones del análisis 📝

Lo que descubrimos sobre el desastre del Titanic, **solo con pandas**:

1. **Datos limpios**: de 891 pasajeros pasamos a 889 después de limpiar (perdimos 2 con `embarked` faltante e imputamos 177 edades con la mediana). Eliminamos la columna `deck` por tener 77% de NaN.

2. **El sexo importa muchísimo**: las mujeres tuvieron una tasa de supervivencia **mucho mayor** que los hombres en **todas** las clases.

3. **La clase también importa**: dentro de cada sexo, los pasajeros de **1ª clase** sobrevivieron más que los de **3ª clase**.

4. **Combinación crítica**: una mujer de 1ª clase tenía ~97% de probabilidad de sobrevivir; un hombre de 3ª, ~14%. Esa diferencia es **enorme** y refleja la política de "*mujeres y niños primero*", que se aplicó **mucho más** entre las clases altas.

5. **Origen de los pasajeros**: ~72% embarcó en **Southampton** (Reino Unido), seguido de Cherbourg y Queenstown.

### Este flujo es lo que harás en cada proyecto de Data Science

```
    Cargar  →  Inspeccionar  →  Limpiar  →  Filtrar  →  Agrupar  →  Concluir
    (NB01)        (NB01)         (NB03)      (NB02)      (NB04)        (tú)
```

Y a veces, además, tendrás que combinar varias tablas (**NB05**).

---

## 8. ¿Y ahora qué? 🚀

¡Felicidades! Has completado los **6 notebooks de prepwork pandas**. Estás listo para el siguiente paso del prepwork: una introducción al **Machine Learning** con `scikit-learn`.

### Repaso final — todas las herramientas que ahora dominas

| Categoría | Métodos |
|---|---|
| Inspección | `.head()`, `.shape`, `.columns`, `.dtypes`, `.info()`, `.describe()` |
| Acceso | `df["col"]`, `df[["c1","c2"]]`, `.iloc`, `.loc` |
| Filtrado | máscaras booleanas, `&`, `|`, `~`, `.isin()` |
| Ordenamiento | `.sort_values()` |
| NaN | `.isna()`, `.dropna()`, `.fillna()` |
| Agrupación | `.groupby()`, `.agg()`, `.value_counts()` |
| Combinación | `pd.concat()`, `.merge()` |

### Recomendación

Antes de pasar a Machine Learning, intenta **repetir este mini-EDA** sobre otro dataset clásico (por ejemplo `tips`, `iris` o el `penguins` que ya conoces) **sin mirar este notebook**. Es la mejor forma de consolidar lo aprendido.

¡Nos vemos en la siguiente parte! 🎓